# Golbal Threshold tuning

In [17]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)


In [18]:
OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA"
)


In [19]:
THRESHOLD_START = 0.01
THRESHOLD_END = 0.99
THRESHOLD_STEP = 0.01

In [20]:
MODEL_LABELS = [
    "MT",   # 0
    "LY",   # 1
    "SP",   # 2
    "ID",   # 3
    "NA",   # 4
    "HI",   # 5
    "IN",   # 6
    "OP",   # 7
    "IP",   # 8
    "it",   # 9
    "ne",   # 10
    "sr",   # 11
    "nb",   # 12
    "re",   # 13
    "en",   # 14
    "ra",   # 15
    "dtp",  # 16
    "fi",   # 17
    "lt",   # 18
    "rv",   # 19
    "ob",   # 20
    "rs",   # 21
    "av",   # 22
    "ds",   # 23
    "ed",   # 24
]


## Load Data

In [21]:
logits_file = OUTPUT_DIR / "dev_logits.npy"
labels_file = OUTPUT_DIR / "dev_labels.npy"

In [22]:
if not logits_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{logits_file}"
    )

if not labels_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{labels_file}"
    )


In [23]:
print("=" * 80)
print("GLOBAL THRESHOLD TUNING")
print("=" * 80)

print("\nLoading Dev logits...")
dev_logits = np.load(logits_file)

print("Loading Dev labels...")
dev_labels = np.load(labels_file)

print("\nLogits shape:", dev_logits.shape)
print("Labels shape:", dev_labels.shape)


if dev_logits.shape != dev_labels.shape:
    raise ValueError(
        "Logits and labels have different shapes:\n"
        f"logits: {dev_logits.shape}\n"
        f"labels: {dev_labels.shape}"
    )

GLOBAL THRESHOLD TUNING

Loading Dev logits...
Loading Dev labels...

Logits shape: (609, 25)
Labels shape: (609, 25)


## LABEL NAMES

In [24]:
num_classes = dev_labels.shape[1]

if MODEL_LABELS is None:

    model_labels = [
        f"class_{i}"
        for i in range(num_classes)
    ]

else:

    model_labels = MODEL_LABELS

    if len(model_labels) != num_classes:
        raise ValueError(
            f"Number of labels ({len(model_labels)}) "
            f"does not match number of classes ({num_classes})."
        )


print("\nNumber of classes:", num_classes)


Number of classes: 25


## LOGITS -> PROBABILITIES

In [25]:
print("\nConverting logits to probabilities...")

# Numerically stable sigmoid
dev_probabilities = np.empty_like(
    dev_logits,
    dtype=np.float64,
)

positive_mask = dev_logits >= 0

dev_probabilities[positive_mask] = (
    1.0
    / (
        1.0
        + np.exp(
            -dev_logits[positive_mask]
        )
    )
)

dev_probabilities[~positive_mask] = (
    np.exp(
        dev_logits[~positive_mask]
    )
    / (
        1.0
        + np.exp(
            dev_logits[~positive_mask]
        )
    )
)



Converting logits to probabilities...


## THRESHOLD SEARCH

In [26]:
thresholds = np.arange(
    THRESHOLD_START,
    THRESHOLD_END + THRESHOLD_STEP / 2,
    THRESHOLD_STEP,
)

results = []


print(
    f"\nTesting {len(thresholds)} thresholds..."
)

for threshold in thresholds:

    predictions = (
        dev_probabilities >= threshold
    ).astype(np.int32)


    macro_f1 = f1_score(
        dev_labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    micro_f1 = f1_score(
        dev_labels,
        predictions,
        average="micro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        dev_labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    macro_precision = precision_score(
        dev_labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    macro_recall = recall_score(
        dev_labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    micro_precision = precision_score(
        dev_labels,
        predictions,
        average="micro",
        zero_division=0,
    )

    micro_recall = recall_score(
        dev_labels,
        predictions,
        average="micro",
        zero_division=0,
    )

    results.append({
        "threshold": threshold,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "weighted_f1": weighted_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
    })


results_df = pd.DataFrame(results)


Testing 99 thresholds...


## FIND BEST THRESHOLD

In [27]:
best_index = results_df[
    "macro_f1"
].idxmax()

best_row = results_df.loc[
    best_index
]

best_threshold = float(
    best_row["threshold"]
)

best_macro_f1 = float(
    best_row["macro_f1"]
)


print("\n" + "=" * 80)
print("BEST GLOBAL THRESHOLD")
print("=" * 80)

print(
    f"\nBest threshold: {best_threshold:.4f}"
)

print(
    f"Macro F1:       {best_row['macro_f1']:.6f}"
)

print(
    f"Micro F1:        {best_row['micro_f1']:.6f}"
)

print(
    f"Weighted F1:     {best_row['weighted_f1']:.6f}"
)

print(
    f"Macro Precision: {best_row['macro_precision']:.6f}"
)

print(
    f"Macro Recall:    {best_row['macro_recall']:.6f}"
)

print(
    f"Micro Precision: {best_row['micro_precision']:.6f}"
)

print(
    f"Micro Recall:    {best_row['micro_recall']:.6f}"
)



BEST GLOBAL THRESHOLD

Best threshold: 0.3100
Macro F1:       0.756381
Micro F1:        0.774291
Weighted F1:     0.770553
Macro Precision: 0.767373
Macro Recall:    0.760157
Micro Precision: 0.770571
Micro Recall:    0.778047


## SAVE ALL THRESHOLD RESULTS

In [28]:
results_file = (
    OUTPUT_DIR
    / "global_threshold_results.csv"
)

results_df.to_csv(
    results_file,
    index=False,
)

print(
    f"\nSaved threshold results to:\n"
    f"{results_file}"
)


Saved threshold results to:
C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA\global_threshold_results.csv


## SAVE BEST THRESHOLD

In [29]:
best_threshold_file = (
    OUTPUT_DIR
    / "best_global_threshold.json"
)

best_threshold_data = {
    "threshold": best_threshold,
    "macro_f1": best_macro_f1,
    "micro_f1": float(
        best_row["micro_f1"]
    ),
    "weighted_f1": float(
        best_row["weighted_f1"]
    ),
    "macro_precision": float(
        best_row["macro_precision"]
    ),
    "macro_recall": float(
        best_row["macro_recall"]
    ),
    "micro_precision": float(
        best_row["micro_precision"]
    ),
    "micro_recall": float(
        best_row["micro_recall"]
    ),
}

with open(
    best_threshold_file,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        best_threshold_data,
        f,
        indent=2,
    )

print(
    f"Saved best threshold to:\n"
    f"{best_threshold_file}"
)



Saved best threshold to:
C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA\best_global_threshold.json


## PER-CLASS PERFORMANCE AT BEST GLOBAL THRESHOLD

In [30]:
best_predictions = (
    dev_probabilities >= best_threshold
).astype(np.int32)


per_class_f1 = f1_score(
    dev_labels,
    best_predictions,
    average=None,
    zero_division=0,
)

per_class_precision = precision_score(
    dev_labels,
    best_predictions,
    average=None,
    zero_division=0,
)

per_class_recall = recall_score(
    dev_labels,
    best_predictions,
    average=None,
    zero_division=0,
)


# Number of actual positives
actual_positives = dev_labels.sum(
    axis=0
)

# Number of predicted positives
predicted_positives = best_predictions.sum(
    axis=0
)


per_class_df = pd.DataFrame({
    "label": model_labels,
    "f1": per_class_f1,
    "precision": per_class_precision,
    "recall": per_class_recall,
    "actual_positives": actual_positives,
    "predicted_positives": predicted_positives,
})


per_class_file = (
    OUTPUT_DIR
    / "global_threshold_per_class_f1.csv"
)

per_class_df.to_csv(
    per_class_file,
    index=False,
)

print(
    f"\nSaved per-class results to:\n"
    f"{per_class_file}"
)


Saved per-class results to:
C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA\global_threshold_per_class_f1.csv


## PLOT

In [31]:
plt.figure(
    figsize=(10, 6)
)

plt.plot(
    results_df["threshold"],
    results_df["macro_f1"],
    label="Macro F1",
    linewidth=2,
)

plt.plot(
    results_df["threshold"],
    results_df["micro_f1"],
    label="Micro F1",
    linewidth=2,
)

plt.plot(
    results_df["threshold"],
    results_df["weighted_f1"],
    label="Weighted F1",
    linewidth=2,
)

plt.axvline(
    best_threshold,
    color="red",
    linestyle="--",
    linewidth=2,
    label=(
        f"Best threshold = "
        f"{best_threshold:.2f}"
    ),
)

plt.xlabel("Global Threshold")
plt.ylabel("F1")
plt.title(
    "Global Threshold Tuning on Dev Set"
)

plt.grid(
    True,
    alpha=0.3,
)

plt.legend()

plt.tight_layout()


plot_file = (
    OUTPUT_DIR
    / "global_threshold_curve.png"
)

plt.savefig(
    plot_file,
    dpi=300,
)

plt.close()

print(
    f"Saved plot to:\n"
    f"{plot_file}"
)


Saved plot to:
C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA\global_threshold_curve.png


In [32]:
print("\n" + "=" * 80)
print("GLOBAL THRESHOLD TUNING COMPLETE")
print("=" * 80)


GLOBAL THRESHOLD TUNING COMPLETE


# Per-class Threshold Tuning

In [33]:
OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA"
)

THRESHOLD_START = 0.01
THRESHOLD_END = 0.99
THRESHOLD_STEP = 0.01

In [34]:
MODEL_LABELS = [
    "MT",   # 0
    "LY",   # 1
    "SP",   # 2
    "ID",   # 3
    "NA",   # 4
    "HI",   # 5
    "IN",   # 6
    "OP",   # 7
    "IP",   # 8
    "it",   # 9
    "ne",   # 10
    "sr",   # 11
    "nb",   # 12
    "re",   # 13
    "en",   # 14
    "ra",   # 15
    "dtp",  # 16
    "fi",   # 17
    "lt",   # 18
    "rv",   # 19
    "ob",   # 20
    "rs",   # 21
    "av",   # 22
    "ds",   # 23
    "ed",   # 24
]


## Load data

In [35]:
logits_file = OUTPUT_DIR / "dev_logits.npy"
labels_file = OUTPUT_DIR / "dev_labels.npy"

if not logits_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{logits_file}"
    )

if not labels_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{labels_file}"
    )


print("=" * 80)
print("PER-CLASS THRESHOLD TUNING")
print("=" * 80)


print("\nLoading Dev logits...")
dev_logits = np.load(logits_file)

print("Loading Dev labels...")
dev_labels = np.load(labels_file)


print("\nLogits shape:", dev_logits.shape)
print("Labels shape:", dev_labels.shape)


if dev_logits.shape != dev_labels.shape:
    raise ValueError(
        "Logits and labels have different shapes:\n"
        f"logits: {dev_logits.shape}\n"
        f"labels: {dev_labels.shape}"
    )



PER-CLASS THRESHOLD TUNING

Loading Dev logits...
Loading Dev labels...

Logits shape: (609, 25)
Labels shape: (609, 25)


## Label names

In [36]:
num_classes = dev_labels.shape[1]

if MODEL_LABELS is None:

    model_labels = [
        f"class_{i}"
        for i in range(num_classes)
    ]

else:

    model_labels = MODEL_LABELS

    if len(model_labels) != num_classes:
        raise ValueError(
            f"Number of labels ({len(model_labels)}) "
            f"does not match number of classes ({num_classes})."
        )


## LOGITS -> PROBABILITIES

In [37]:
print("\nConverting logits to probabilities...")

dev_probabilities = np.empty_like(
    dev_logits,
    dtype=np.float64,
)

positive_mask = dev_logits >= 0

dev_probabilities[positive_mask] = (
    1.0
    / (
        1.0
        + np.exp(
            -dev_logits[positive_mask]
        )
    )
)

dev_probabilities[~positive_mask] = (
    np.exp(
        dev_logits[~positive_mask]
    )
    / (
        1.0
        + np.exp(
            dev_logits[~positive_mask]
        )
    )
)



Converting logits to probabilities...


## THRESHOLDS TO TEST

In [38]:
thresholds = np.arange(
    THRESHOLD_START,
    THRESHOLD_END + THRESHOLD_STEP / 2,
    THRESHOLD_STEP,
)

## FIND BEST THRESHOLD FOR EACH CLASS

In [39]:
optimal_thresholds = []

per_class_results = []


print("\n" + "=" * 80)
print("SEARCHING FOR OPTIMAL THRESHOLD FOR EACH CLASS")
print("=" * 80)


for class_idx in range(num_classes):

    label_name = model_labels[class_idx]

    y_true = dev_labels[
        :, class_idx
    ]

    y_prob = dev_probabilities[
        :, class_idx
    ]


    best_f1 = -1.0
    best_threshold = 0.5
    best_precision = 0.0
    best_recall = 0.0


    for threshold in thresholds:

        y_pred = (
            y_prob >= threshold
        ).astype(np.int32)


        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )

        precision = precision_score(
            y_true,
            y_pred,
            zero_division=0,
        )

        recall = recall_score(
            y_true,
            y_pred,
            zero_division=0,
        )


        if f1 > best_f1:

            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall


    actual_positives = int(
        y_true.sum()
    )

    predictions_at_best = (
        y_prob >= best_threshold
    ).astype(np.int32)

    predicted_positives = int(
        predictions_at_best.sum()
    )


    optimal_thresholds.append(
        best_threshold
    )


    per_class_results.append({

        "class_index": class_idx,

        "label": label_name,

        "optimal_threshold": best_threshold,

        "f1": best_f1,

        "precision": best_precision,

        "recall": best_recall,

        "actual_positives": actual_positives,

        "predicted_positives": predicted_positives,

    })


    print(
        f"{class_idx:3d} | "
        f"{label_name:30s} | "
        f"threshold={best_threshold:.2f} | "
        f"F1={best_f1:.4f} | "
        f"P={best_precision:.4f} | "
        f"R={best_recall:.4f}"
    )


per_class_df = pd.DataFrame(
    per_class_results
)




SEARCHING FOR OPTIMAL THRESHOLD FOR EACH CLASS
  0 | MT                             | threshold=0.28 | F1=0.8254 | P=0.8125 | R=0.8387
  1 | LY                             | threshold=0.16 | F1=0.8817 | P=0.9111 | R=0.8542
  2 | SP                             | threshold=0.36 | F1=0.8547 | P=0.8772 | R=0.8333
  3 | ID                             | threshold=0.29 | F1=0.6931 | P=0.8333 | R=0.5932
  4 | NA                             | threshold=0.57 | F1=0.8443 | P=0.9195 | R=0.7805
  5 | HI                             | threshold=0.28 | F1=0.8067 | P=0.8421 | R=0.7742
  6 | IN                             | threshold=0.40 | F1=0.8496 | P=0.8496 | R=0.8496
  7 | OP                             | threshold=0.35 | F1=0.7970 | P=0.7950 | R=0.7990
  8 | IP                             | threshold=0.38 | F1=0.8101 | P=0.8348 | R=0.7869
  9 | it                             | threshold=0.32 | F1=0.7576 | P=0.7353 | R=0.7812
 10 | ne                             | threshold=0.40 | F1=0.8481 | P=0.

## SAVE PER-CLASS THRESHOLDS

In [40]:
threshold_file = (
    OUTPUT_DIR
    / "per_class_thresholds.csv"
)

per_class_df.to_csv(
    threshold_file,
    index=False,
)


print(
    f"\nSaved per-class thresholds to:\n"
    f"{threshold_file}"
)


# JSON version

threshold_dict = {
    model_labels[i]: float(
        optimal_thresholds[i]
    )
    for i in range(num_classes)
}


threshold_json_file = (
    OUTPUT_DIR
    / "per_class_thresholds.json"
)

with open(
    threshold_json_file,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        threshold_dict,
        f,
        indent=2,
    )


print(
    f"Saved JSON thresholds to:\n"
    f"{threshold_json_file}"
)


Saved per-class thresholds to:
C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA\per_class_thresholds.csv
Saved JSON thresholds to:
C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA\per_class_thresholds.json


## APPLY PER-CLASS THRESHOLDS

In [41]:

optimal_thresholds_array = np.asarray(
    optimal_thresholds
)


per_class_predictions = (
    dev_probabilities
    >= optimal_thresholds_array
).astype(np.int32)


## OVERALL PERFORMANCE

In [42]:
macro_f1 = f1_score(
    dev_labels,
    per_class_predictions,
    average="macro",
    zero_division=0,
)

micro_f1 = f1_score(
    dev_labels,
    per_class_predictions,
    average="micro",
    zero_division=0,
)

weighted_f1 = f1_score(
    dev_labels,
    per_class_predictions,
    average="weighted",
    zero_division=0,
)

macro_precision = precision_score(
    dev_labels,
    per_class_predictions,
    average="macro",
    zero_division=0,
)

macro_recall = recall_score(
    dev_labels,
    per_class_predictions,
    average="macro",
    zero_division=0,
)

micro_precision = precision_score(
    dev_labels,
    per_class_predictions,
    average="micro",
    zero_division=0,
)

micro_recall = recall_score(
    dev_labels,
    per_class_predictions,
    average="micro",
    zero_division=0,
)


In [43]:
print("\n" + "=" * 80)
print("PER-CLASS THRESHOLD RESULTS")
print("=" * 80)

print(
    f"\nMacro F1:       {macro_f1:.6f}"
)

print(
    f"Micro F1:        {micro_f1:.6f}"
)

print(
    f"Weighted F1:     {weighted_f1:.6f}"
)

print(
    f"Macro Precision: {macro_precision:.6f}"
)

print(
    f"Macro Recall:    {macro_recall:.6f}"
)

print(
    f"Micro Precision: {micro_precision:.6f}"
)

print(
    f"Micro Recall:    {micro_recall:.6f}"
)



PER-CLASS THRESHOLD RESULTS

Macro F1:       0.779304
Micro F1:        0.788940
Weighted F1:     0.789643
Macro Precision: 0.786219
Macro Recall:    0.781995
Micro Precision: 0.799502
Micro Recall:    0.778654


## SAVE SUMMARY

In [44]:

summary_df = pd.DataFrame([{

    "method": "per_class_thresholds",

    "macro_f1": macro_f1,

    "micro_f1": micro_f1,

    "weighted_f1": weighted_f1,

    "macro_precision": macro_precision,

    "macro_recall": macro_recall,

    "micro_precision": micro_precision,

    "micro_recall": micro_recall,

}])


summary_file = (
    OUTPUT_DIR
    / "per_class_threshold_summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False,
)


print(
    f"\nSaved summary to:\n"
    f"{summary_file}"
)




Saved summary to:
C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA\per_class_threshold_summary.csv


## F1 CURVES FOR EACH CLASS

In [47]:
fig, ax = plt.subplots(
    figsize=(16, 9)
)

for class_idx in range(num_classes):

    y_true = dev_labels[:, class_idx]

    y_prob = dev_probabilities[:, class_idx]

    class_f1_scores = []

    for threshold in thresholds:

        y_pred = (
            y_prob >= threshold
        ).astype(np.int32)

        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )

        class_f1_scores.append(f1)

    ax.plot(
        thresholds,
        class_f1_scores,
        linewidth=1.5,
        alpha=0.7,
        label=model_labels[class_idx],
    )


ax.set_xlabel(
    "Threshold"
)

ax.set_ylabel(
    "F1"
)

ax.set_title(
    "Per-Class F1 vs Threshold"
)

ax.grid(
    True,
    alpha=0.3,
)

# Legend outside the plot
ax.legend(
    title="Class",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    fontsize=9,
    title_fontsize=10,
    frameon=True,
)

# Leave enough space for the external legend
plt.tight_layout(
    rect=[0, 0, 0.82, 1]
)


plot_file = (
    OUTPUT_DIR
    / "per_class_threshold_f1_curve.png"
)

plt.savefig(
    plot_file,
    dpi=300,
    bbox_inches="tight",
)

plt.close()


print(
    f"Saved F1 curves to:\n"
    f"{plot_file}"
)

Saved F1 curves to:
C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA\per_class_threshold_f1_curve.png


In [46]:
print("\n" + "=" * 80)
print("PER-CLASS THRESHOLD TUNING COMPLETE")
print("=" * 80)


PER-CLASS THRESHOLD TUNING COMPLETE


# parent - child comparison

In [2]:
# ============================================================
# Model 3
# Hierarchical XLM-R + Parent-Specific Experts
#
# Threshold analysis:
#
#   1. Global threshold
#   2. Per-class thresholds
#   3. Fixed threshold = 0.50
#
# Reports:
#
#   Parent Macro F1
#   Parent Micro F1
#
#   Child Macro F1
#   Child Micro F1
#
#   Overall Macro F1
#   Overall Micro F1
#   Overall Weighted F1
# ============================================================


# ============================================================
# Imports
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)


# ============================================================
# Configuration
# ============================================================

OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Desktop\XLMR_Threshold_tuning\Combined_hybrid_no_NA"
)


# ------------------------------------------------------------
# Threshold search range
# ------------------------------------------------------------

THRESHOLD_START = 0.01
THRESHOLD_END = 0.99
THRESHOLD_STEP = 0.01


# ------------------------------------------------------------
# Fixed threshold
# ------------------------------------------------------------

FIXED_THRESHOLD = 0.50


# ============================================================
# Model 3 label order
#
# This MUST exactly match the order used by the model.
# ============================================================

MODEL_LABELS = [

    # --------------------------------------------------------
    # Parents
    # --------------------------------------------------------

    "MT",   # 0
    "LY",   # 1
    "SP",   # 2
    "ID",   # 3
    "NA",   # 4
    "HI",   # 5
    "IN",   # 6
    "OP",   # 7
    "IP",   # 8

    # --------------------------------------------------------
    # Children
    # --------------------------------------------------------

    "it",   # 9

    "ne",   # 10
    "sr",   # 11
    "nb",   # 12

    "re",   # 13

    "en",   # 14
    "ra",   # 15
    "dtp",  # 16
    "fi",   # 17
    "lt",   # 18

    "rv",   # 19
    "ob",   # 20
    "rs",   # 21
    "av",   # 22

    "ds",   # 23
    "ed",   # 24
]


# ============================================================
# Hierarchy
# ============================================================

PARENTS = [

    "MT",
    "LY",
    "SP",
    "ID",
    "NA",
    "HI",
    "IN",
    "OP",
    "IP",

]


CHILDREN = [

    "it",

    "ne",
    "sr",
    "nb",

    "re",

    "en",
    "ra",
    "dtp",
    "fi",
    "lt",

    "rv",
    "ob",
    "rs",
    "av",

    "ds",
    "ed",

]


NUM_PARENTS = len(PARENTS)
NUM_CHILDREN = len(CHILDREN)
NUM_CLASSES = NUM_PARENTS + NUM_CHILDREN


# ============================================================
# Verify label configuration
# ============================================================

if len(MODEL_LABELS) != NUM_CLASSES:

    raise ValueError(

        f"MODEL_LABELS contains "
        f"{len(MODEL_LABELS)} labels, "

        f"but expected {NUM_CLASSES}."

    )


if MODEL_LABELS != PARENTS + CHILDREN:

    raise ValueError(

        "MODEL_LABELS does not match "
        "PARENTS + CHILDREN order."

    )


# ============================================================
# Files
# ============================================================

LOGITS_FILE = (
    OUTPUT_DIR
    / "dev_logits.npy"
)


LABELS_FILE = (
    OUTPUT_DIR
    / "dev_labels.npy"
)


# ============================================================
# Load DEV data
# ============================================================

print("\n" + "=" * 80)
print("HIERARCHICAL THRESHOLD ANALYSIS")
print("=" * 80)


if not LOGITS_FILE.exists():

    raise FileNotFoundError(
        f"Could not find:\n{LOGITS_FILE}"
    )


if not LABELS_FILE.exists():

    raise FileNotFoundError(
        f"Could not find:\n{LABELS_FILE}"
    )


print("\nLoading DEV logits...")

dev_logits = np.load(
    LOGITS_FILE
)


print("Loading DEV labels...")

dev_labels = np.load(
    LABELS_FILE
)


print(
    "\nLogits shape:",
    dev_logits.shape
)


print(
    "Labels shape:",
    dev_labels.shape
)


# ============================================================
# Verify shapes
# ============================================================

if dev_logits.shape != dev_labels.shape:

    raise ValueError(

        "Logits and labels have different shapes:\n"

        f"logits: {dev_logits.shape}\n"

        f"labels: {dev_labels.shape}"

    )


if dev_logits.shape[1] != NUM_CLASSES:

    raise ValueError(

        f"Expected {NUM_CLASSES} output classes, "

        f"but found {dev_logits.shape[1]}."

    )


# ============================================================
# Convert logits -> probabilities
#
# Numerically stable sigmoid
# ============================================================

print(
    "\nConverting logits to probabilities..."
)


dev_probabilities = np.empty_like(

    dev_logits,

    dtype=np.float64,

)


positive_mask = (
    dev_logits >= 0
)


dev_probabilities[positive_mask] = (

    1.0
    /
    (
        1.0
        +
        np.exp(
            -dev_logits[positive_mask]
        )
    )

)


dev_probabilities[~positive_mask] = (

    np.exp(
        dev_logits[~positive_mask]
    )
    /
    (
        1.0
        +
        np.exp(
            dev_logits[~positive_mask]
        )
    )

)


# ============================================================
# Thresholds to test
# ============================================================

thresholds = np.arange(

    THRESHOLD_START,

    THRESHOLD_END
    +
    THRESHOLD_STEP / 2,

    THRESHOLD_STEP,

)


print(
    "\nNumber of thresholds:",
    len(thresholds)
)


# ============================================================
# Metric function
# ============================================================

def calculate_metrics(

    labels,

    predictions,

):

    # --------------------------------------------------------
    # Parent
    # --------------------------------------------------------

    parent_labels = labels[
        :,
        :NUM_PARENTS
    ]


    parent_predictions = predictions[
        :,
        :NUM_PARENTS
    ]


    parent_macro_f1 = f1_score(

        parent_labels,

        parent_predictions,

        average="macro",

        zero_division=0,

    )


    parent_micro_f1 = f1_score(

        parent_labels,

        parent_predictions,

        average="micro",

        zero_division=0,

    )


    # --------------------------------------------------------
    # Children
    # --------------------------------------------------------

    child_labels = labels[
        :,
        NUM_PARENTS:
    ]


    child_predictions = predictions[
        :,
        NUM_PARENTS:
    ]


    child_macro_f1 = f1_score(

        child_labels,

        child_predictions,

        average="macro",

        zero_division=0,

    )


    child_micro_f1 = f1_score(

        child_labels,

        child_predictions,

        average="micro",

        zero_division=0,

    )


    # --------------------------------------------------------
    # Overall
    # --------------------------------------------------------

    overall_macro_f1 = f1_score(

        labels,

        predictions,

        average="macro",

        zero_division=0,

    )


    overall_micro_f1 = f1_score(

        labels,

        predictions,

        average="micro",

        zero_division=0,

    )


    overall_weighted_f1 = f1_score(

        labels,

        predictions,

        average="weighted",

        zero_division=0,

    )


    return {

        "parent_macro_f1":
            float(parent_macro_f1),

        "parent_micro_f1":
            float(parent_micro_f1),

        "child_macro_f1":
            float(child_macro_f1),

        "child_micro_f1":
            float(child_micro_f1),

        "overall_macro_f1":
            float(overall_macro_f1),

        "overall_micro_f1":
            float(overall_micro_f1),

        "overall_weighted_f1":
            float(overall_weighted_f1),

    }


# ============================================================
# 1. GLOBAL THRESHOLD
#
# Find one threshold for all 25 classes.
#
# IMPORTANT:
# We select the threshold using OVERALL MACRO F1,
# matching your previous implementation.
# ============================================================

print("\n" + "=" * 80)
print("SEARCHING GLOBAL THRESHOLD")
print("=" * 80)


global_results = []


for threshold in thresholds:

    predictions = (

        dev_probabilities
        >= threshold

    ).astype(np.int32)


    metrics = calculate_metrics(

        dev_labels,

        predictions,

    )


    global_results.append({

        "threshold":
            float(threshold),

        **metrics,

    })


global_results_df = pd.DataFrame(
    global_results
)


# ============================================================
# Find best global threshold
# ============================================================

best_global_index = (

    global_results_df[
        "overall_macro_f1"
    ].idxmax()

)


best_global_row = (

    global_results_df.loc[
        best_global_index
    ]

)


best_global_threshold = float(

    best_global_row[
        "threshold"
    ]

)


# ============================================================
# Print global result
# ============================================================

print("\n" + "=" * 80)
print(
    f"Global Threshold "
    f"({best_global_threshold:.2f})"
)
print("=" * 80)


print(
    f"Parent Macro F1   : "
    f"{best_global_row['parent_macro_f1']:.4f}"
)


print(
    f"Parent Micro F1   : "
    f"{best_global_row['parent_micro_f1']:.4f}"
)


print(
    f"\nChild Macro F1    : "
    f"{best_global_row['child_macro_f1']:.4f}"
)


print(
    f"Child Micro F1    : "
    f"{best_global_row['child_micro_f1']:.4f}"
)


print(
    f"\nOverall Macro F1  : "
    f"{best_global_row['overall_macro_f1']:.4f}"
)


print(
    f"Overall Micro F1  : "
    f"{best_global_row['overall_micro_f1']:.4f}"
)


print(
    f"Overall Weighted F1: "
    f"{best_global_row['overall_weighted_f1']:.4f}"
)


# ============================================================
# Save global threshold results
# ============================================================

global_results_file = (

    OUTPUT_DIR
    / "hierarchical_global_threshold_results.csv"

)


global_results_df.to_csv(

    global_results_file,

    index=False,

)


# ============================================================
# 2. PER-CLASS THRESHOLDS
#
# Find one threshold independently for each of the
# 25 classes.
#
# Each class threshold is selected by that class's F1.
# ============================================================

print("\n" + "=" * 80)
print("SEARCHING PER-CLASS THRESHOLDS")
print("=" * 80)


optimal_thresholds = []


per_class_results = []


for class_idx in range(NUM_CLASSES):

    label_name = (
        MODEL_LABELS[class_idx]
    )


    y_true = (

        dev_labels[
            :,
            class_idx
        ]

    )


    y_prob = (

        dev_probabilities[
            :,
            class_idx
        ]

    )


    best_f1 = -1.0

    best_threshold = 0.50

    best_precision = 0.0

    best_recall = 0.0


    # --------------------------------------------------------
    # Search thresholds
    # --------------------------------------------------------

    for threshold in thresholds:

        y_pred = (

            y_prob
            >= threshold

        ).astype(np.int32)


        f1 = f1_score(

            y_true,

            y_pred,

            zero_division=0,

        )


        precision = precision_score(

            y_true,

            y_pred,

            zero_division=0,

        )


        recall = recall_score(

            y_true,

            y_pred,

            zero_division=0,

        )


        if f1 > best_f1:

            best_f1 = f1

            best_threshold = (
                threshold
            )

            best_precision = (
                precision
            )

            best_recall = (
                recall
            )


    optimal_thresholds.append(

        best_threshold

    )


    actual_positives = int(

        y_true.sum()

    )


    predictions_at_best = (

        y_prob
        >= best_threshold

    ).astype(np.int32)


    predicted_positives = int(

        predictions_at_best.sum()

    )


    per_class_results.append({

        "class_index":
            class_idx,

        "label":
            label_name,

        "type":
            (
                "parent"
                if class_idx < NUM_PARENTS
                else "child"
            ),

        "optimal_threshold":
            float(best_threshold),

        "f1":
            float(best_f1),

        "precision":
            float(best_precision),

        "recall":
            float(best_recall),

        "actual_positives":
            actual_positives,

        "predicted_positives":
            predicted_positives,

    })


    print(

        f"{class_idx:2d} | "

        f"{label_name:5s} | "

        f"{'parent' if class_idx < NUM_PARENTS else 'child':6s} | "

        f"threshold={best_threshold:.2f} | "

        f"F1={best_f1:.4f} | "

        f"P={best_precision:.4f} | "

        f"R={best_recall:.4f}"

    )


per_class_results_df = pd.DataFrame(

    per_class_results

)


# ============================================================
# Apply per-class thresholds
# ============================================================

optimal_thresholds_array = np.asarray(

    optimal_thresholds,

    dtype=np.float64,

)


per_class_predictions = (

    dev_probabilities
    >= optimal_thresholds_array

).astype(np.int32)


# ============================================================
# Calculate per-class threshold metrics
# ============================================================

per_class_metrics = calculate_metrics(

    dev_labels,

    per_class_predictions,

)


# ============================================================
# Print per-class result
# ============================================================

print("\n" + "=" * 80)
print("Per-class Thresholds")
print("=" * 80)


print(
    f"Parent Macro F1   : "
    f"{per_class_metrics['parent_macro_f1']:.4f}"
)


print(
    f"Parent Micro F1   : "
    f"{per_class_metrics['parent_micro_f1']:.4f}"
)


print(
    f"\nChild Macro F1    : "
    f"{per_class_metrics['child_macro_f1']:.4f}"
)


print(
    f"Child Micro F1    : "
    f"{per_class_metrics['child_micro_f1']:.4f}"
)


print(
    f"\nOverall Macro F1  : "
    f"{per_class_metrics['overall_macro_f1']:.4f}"
)


print(
    f"Overall Micro F1  : "
    f"{per_class_metrics['overall_micro_f1']:.4f}"
)


print(
    f"Overall Weighted F1: "
    f"{per_class_metrics['overall_weighted_f1']:.4f}"
)


# ============================================================
# 3. FIXED THRESHOLD = 0.50
# ============================================================

fixed_predictions = (

    dev_probabilities
    >= FIXED_THRESHOLD

).astype(np.int32)


fixed_metrics = calculate_metrics(

    dev_labels,

    fixed_predictions,

)


# ============================================================
# Print fixed threshold result
# ============================================================

print("\n" + "=" * 80)
print(
    f"Threshold = "
    f"{FIXED_THRESHOLD:.2f}"
)
print("=" * 80)


print(
    f"Parent Macro F1   : "
    f"{fixed_metrics['parent_macro_f1']:.4f}"
)


print(
    f"Parent Micro F1   : "
    f"{fixed_metrics['parent_micro_f1']:.4f}"
)


print(
    f"\nChild Macro F1    : "
    f"{fixed_metrics['child_macro_f1']:.4f}"
)


print(
    f"Child Micro F1    : "
    f"{fixed_metrics['child_micro_f1']:.4f}"
)


print(
    f"\nOverall Macro F1  : "
    f"{fixed_metrics['overall_macro_f1']:.4f}"
)


print(
    f"Overall Micro F1  : "
    f"{fixed_metrics['overall_micro_f1']:.4f}"
)


print(
    f"Overall Weighted F1: "
    f"{fixed_metrics['overall_weighted_f1']:.4f}"
)


# ============================================================
# COMPARISON TABLE
# ============================================================

comparison_df = pd.DataFrame([

    {

        "method":
            f"Global Threshold "
            f"({best_global_threshold:.2f})",

        **{

            key.replace(
                "_f1",
                ""
            ):
            value

            for key, value
            in {}

        },

        "parent_macro_f1":
            float(
                best_global_row[
                    "parent_macro_f1"
                ]
            ),

        "parent_micro_f1":
            float(
                best_global_row[
                    "parent_micro_f1"
                ]
            ),

        "child_macro_f1":
            float(
                best_global_row[
                    "child_macro_f1"
                ]
            ),

        "child_micro_f1":
            float(
                best_global_row[
                    "child_micro_f1"
                ]
            ),

        "overall_macro_f1":
            float(
                best_global_row[
                    "overall_macro_f1"
                ]
            ),

        "overall_micro_f1":
            float(
                best_global_row[
                    "overall_micro_f1"
                ]
            ),

        "overall_weighted_f1":
            float(
                best_global_row[
                    "overall_weighted_f1"
                ]
            ),

    },

    {

        "method":
            "Per-class Thresholds",

        **per_class_metrics,

    },

    {

        "method":
            f"Threshold = "
            f"{FIXED_THRESHOLD:.2f}",

        **fixed_metrics,

    },

])


# ============================================================
# Print comparison table
# ============================================================

print("\n" + "=" * 100)
print("THRESHOLD COMPARISON")
print("=" * 100)


print(

    comparison_df.to_string(

        index=False,

        float_format=lambda x:
            f"{x:.4f}"

    )

)


# ============================================================
# Save comparison
# ============================================================

comparison_file = (

    OUTPUT_DIR
    / "hierarchical_threshold_comparison.csv"

)


comparison_df.to_csv(

    comparison_file,

    index=False,

)


# ============================================================
# Save per-class thresholds
# ============================================================

threshold_file = (

    OUTPUT_DIR
    / "hierarchical_per_class_thresholds.csv"

)


per_class_results_df.to_csv(

    threshold_file,

    index=False,

)


threshold_json_file = (

    OUTPUT_DIR
    / "hierarchical_per_class_thresholds.json"

)


threshold_dict = {

    MODEL_LABELS[i]:
        float(optimal_thresholds[i])

    for i in range(NUM_CLASSES)

}


with open(

    threshold_json_file,

    "w",

    encoding="utf-8",

) as f:

    json.dump(

        threshold_dict,

        f,

        indent=2,

    )


# ============================================================
# Save best global threshold
# ============================================================

global_threshold_json = (

    OUTPUT_DIR
    / "hierarchical_best_global_threshold.json"

)


global_threshold_data = {

    "threshold":
        best_global_threshold,

    "selection_metric":
        "overall_macro_f1",

    "parent_macro_f1":
        float(
            best_global_row[
                "parent_macro_f1"
            ]
        ),

    "parent_micro_f1":
        float(
            best_global_row[
                "parent_micro_f1"
            ]
        ),

    "child_macro_f1":
        float(
            best_global_row[
                "child_macro_f1"
            ]
        ),

    "child_micro_f1":
        float(
            best_global_row[
                "child_micro_f1"
            ]
        ),

    "overall_macro_f1":
        float(
            best_global_row[
                "overall_macro_f1"
            ]
        ),

    "overall_micro_f1":
        float(
            best_global_row[
                "overall_micro_f1"
            ]
        ),

    "overall_weighted_f1":
        float(
            best_global_row[
                "overall_weighted_f1"
            ]
        ),

}


with open(

    global_threshold_json,

    "w",

    encoding="utf-8",

) as f:

    json.dump(

        global_threshold_data,

        f,

        indent=2,

    )


# ============================================================
# PLOT GLOBAL THRESHOLD
# ============================================================

plt.figure(

    figsize=(10, 6)

)


plt.plot(

    global_results_df[
        "threshold"
    ],

    global_results_df[
        "overall_macro_f1"
    ],

    label="Overall Macro F1",

    linewidth=2,

)


plt.plot(

    global_results_df[
        "threshold"
    ],

    global_results_df[
        "overall_micro_f1"
    ],

    label="Overall Micro F1",

    linewidth=2,

)


plt.plot(

    global_results_df[
        "threshold"
    ],

    global_results_df[
        "child_macro_f1"
    ],

    label="Child Macro F1",

    linewidth=2,

)


plt.plot(

    global_results_df[
        "threshold"
    ],

    global_results_df[
        "parent_macro_f1"
    ],

    label="Parent Macro F1",

    linewidth=2,

)


plt.axvline(

    best_global_threshold,

    color="red",

    linestyle="--",

    linewidth=2,

    label=(

        f"Best global threshold = "
        f"{best_global_threshold:.2f}"

    ),

)


plt.xlabel(
    "Global Threshold"
)


plt.ylabel(
    "F1"
)


plt.title(
    "Hierarchical Model 3 — Global Threshold Tuning"
)


plt.grid(
    True,
    alpha=0.3,
)


plt.legend()


plt.tight_layout()


global_plot_file = (

    OUTPUT_DIR
    / "hierarchical_global_threshold_curve.png"

)


plt.savefig(

    global_plot_file,

    dpi=300,

)


plt.close()


# ============================================================
# PLOT PER-CLASS THRESHOLDS
# ============================================================

fig, ax = plt.subplots(

    figsize=(16, 9)

)


for class_idx in range(NUM_CLASSES):

    y_true = (
        dev_labels[:, class_idx]
    )

    y_prob = (
        dev_probabilities[:, class_idx]
    )


    class_f1_scores = []


    for threshold in thresholds:

        y_pred = (

            y_prob
            >= threshold

        ).astype(np.int32)


        f1 = f1_score(

            y_true,

            y_pred,

            zero_division=0,

        )


        class_f1_scores.append(
            f1
        )


    ax.plot(

        thresholds,

        class_f1_scores,

        linewidth=1.5,

        alpha=0.7,

        label=MODEL_LABELS[class_idx],

    )


ax.set_xlabel(
    "Threshold"
)


ax.set_ylabel(
    "F1"
)


ax.set_title(
    "Model 3 — Per-Class F1 vs Threshold"
)


ax.grid(
    True,
    alpha=0.3,
)


ax.legend(

    title="Class",

    loc="center left",

    bbox_to_anchor=(1.02, 0.5),

    fontsize=9,

    title_fontsize=10,

)


plt.tight_layout(

    rect=[0, 0, 0.82, 1]

)


per_class_plot_file = (

    OUTPUT_DIR
    / "hierarchical_per_class_threshold_f1_curve.png"

)


plt.savefig(

    per_class_plot_file,

    dpi=300,

    bbox_inches="tight",

)


plt.close()


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 100)
print("THRESHOLD ANALYSIS COMPLETE")
print("=" * 100)


print(
    "\nBest global threshold:",
    f"{best_global_threshold:.2f}"
)


print(
    "\nSaved comparison:",
    comparison_file
)


print(
    "\nSaved global threshold:",
    global_threshold_json
)


print(
    "\nSaved per-class thresholds:",
    threshold_json_file
)


print(
    "\nSaved global threshold plot:",
    global_plot_file
)


print(
    "\nSaved per-class threshold plot:",
    per_class_plot_file
)


print("\n" + "=" * 100)



HIERARCHICAL THRESHOLD ANALYSIS

Loading DEV logits...
Loading DEV labels...

Logits shape: (609, 25)
Labels shape: (609, 25)

Converting logits to probabilities...

Number of thresholds: 99

SEARCHING GLOBAL THRESHOLD

Global Threshold (0.31)
Parent Macro F1   : 0.8033
Parent Micro F1   : 0.8122

Child Macro F1    : 0.7300
Child Micro F1    : 0.7144

Overall Macro F1  : 0.7564
Overall Micro F1  : 0.7743
Overall Weighted F1: 0.7706

SEARCHING PER-CLASS THRESHOLDS
 0 | MT    | parent | threshold=0.28 | F1=0.8254 | P=0.8125 | R=0.8387
 1 | LY    | parent | threshold=0.16 | F1=0.8817 | P=0.9111 | R=0.8542
 2 | SP    | parent | threshold=0.36 | F1=0.8547 | P=0.8772 | R=0.8333
 3 | ID    | parent | threshold=0.29 | F1=0.6931 | P=0.8333 | R=0.5932
 4 | NA    | parent | threshold=0.57 | F1=0.8443 | P=0.9195 | R=0.7805
 5 | HI    | parent | threshold=0.28 | F1=0.8067 | P=0.8421 | R=0.7742
 6 | IN    | parent | threshold=0.40 | F1=0.8496 | P=0.8496 | R=0.8496
 7 | OP    | parent | threshold=0.